# Module 8.2 — RAG as a Tool for Agents

`create_retriever_tool` wraps any retriever as a LangChain tool that agents can call.
Agents decide **when** to retrieve vs when to respond directly.

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.schema import Document
from langchain.tools.retriever import create_retriever_tool
from langchain.agents import create_react_agent, AgentExecutor
from langchain_core.prompts import PromptTemplate

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm        = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# ── Knowledge Base 1: AI Models ───────────────────────────────────────────────
ai_docs = [
    Document(page_content="GPT-4o is OpenAI's flagship multimodal model released in May 2024."),
    Document(page_content="Claude 3.5 Sonnet is Anthropic's model known for coding and reasoning."),
    Document(page_content="Gemini 1.5 Pro supports a 1 million token context window."),
]
ai_vs   = Chroma.from_documents(ai_docs, embeddings, collection_name="ai_models")
ai_tool = create_retriever_tool(
    ai_vs.as_retriever(search_kwargs={"k": 2}),
    name="ai_models_search",
    description="Search for information about AI language models like GPT-4, Claude, Gemini."
)

# ── Knowledge Base 2: Programming Languages ───────────────────────────────────
code_docs = [
    Document(page_content="Python supports list comprehensions, decorators, and context managers."),
    Document(page_content="Rust ensures memory safety through its ownership and borrowing system."),
    Document(page_content="Go uses goroutines for lightweight concurrency."),
]
code_vs   = Chroma.from_documents(code_docs, embeddings, collection_name="programming")
code_tool = create_retriever_tool(
    code_vs.as_retriever(search_kwargs={"k": 2}),
    name="programming_search",
    description="Search for information about programming languages: Python, Rust, Go, etc."
)

tools = [ai_tool, code_tool]

# ── ReAct Agent ───────────────────────────────────────────────────────────────
react_prompt = PromptTemplate.from_template("""
Answer the question using the available tools.

Tools: {tools}
Tool names: {tool_names}

Question: {input}
{agent_scratchpad}
""")

agent   = create_react_agent(llm, tools, react_prompt)
executor = AgentExecutor(agent=agent, tools=tools, verbose=True, max_iterations=4)

result = executor.invoke({"input": "What makes Claude 3.5 Sonnet stand out, and how does Python handle concurrency?"})
print("\nFinal Answer:", result["output"])
